# FREUID — training on VESSL workspace

Runs directly on the VESSL GPU — no Colab, no ngrok needed.

**How to connect from your IDE:**
1. Top-right kernel picker → `Select Another Kernel...` → `Existing Jupyter Server...`
2. Paste the VESSL Jupyter URL (from the workspace page, looks like `https://jupyter-wsp-....betelgeuse.cloud.vessl.ai/?token=...`)
3. Select `Python 3`

All cells run on the VESSL machine. Checkpoints and submissions are saved to `/root/repo/checkpoints/` and `/root/repo/submissions/`.

In [ ]:
# Environment check — GPU, data, package
import torch, os
from pathlib import Path

print('device  :', 'cuda -', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (no GPU detected!)')
print('cwd     :', os.getcwd())

data = Path('data')
train_imgs = list((data / 'train/train').glob('*.jpeg'))
test_imgs  = list((data / 'public_test/public_test').glob('*.jpeg'))
print(f'train   : {len(train_imgs)} images')
print(f'test    : {len(test_imgs)} images')

import freuid
print('freuid  : OK')

In [ ]:
# Train — change CONFIG to any configs/*.yaml
CONFIG = 'configs/baseline_v0.yaml'
!python -m freuid.train --config {CONFIG}

In [ ]:
# Inference → submission CSV
# Checkpoint name comes from the config's `name:` field
CKPT = 'checkpoints/baseline_v0.pt'
OUT  = 'submissions/baseline_v0.csv'
!python -m freuid.infer --checkpoint {CKPT} --out {OUT}
!head -3 {OUT} && echo '...' && wc -l {OUT}

In [ ]:
# (optional) Submit to Kaggle directly from the workspace
OUT = 'submissions/baseline_v0.csv'
MSG = 'baseline_v0 recapture+probe, VESSL'
!kaggle competitions submit -c the-freuid-challenge-2026-ijcai-ecai -f {OUT} -m "{MSG}"